# Credit Risk Modelling & Portfolio Stress Testing
### LendingClub 2007–2018Q4

**Objective:** build an end-to-end credit-risk workflow: PD modelling → validation/calibration → LGD/EAD → ECL → Monte Carlo portfolio loss → VaR/Expected Shortfall → stress testing.

> This is an academic portfolio project. EAD and LGD are simplified proxies and are explicitly treated as modelling assumptions.

## 1. Business problem

A lender wants to answer four questions:

1. **Who is likely to default?** → PD
2. **How much is likely to be lost if default occurs?** → LGD
3. **How much is exposed?** → EAD
4. **What could happen to the whole portfolio in bad scenarios?** → Monte Carlo, VaR, ES, stress testing

\[
ECL = PD \times LGD \times EAD
\]

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, brier_score_loss, mean_absolute_error
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from scipy.stats import norm

PATH = "accepted_2007_to_2018Q4.csv.gz"
USECOLS = [
    "loan_amnt","funded_amnt","term","int_rate","installment","grade","sub_grade",
    "emp_length","home_ownership","annual_inc","verification_status","purpose","dti",
    "delinq_2yrs","fico_range_low","fico_range_high","inq_last_6mths","open_acc",
    "pub_rec","revol_bal","revol_util","total_acc","pub_rec_bankruptcies",
    "issue_d","loan_status","recoveries"
]

# Chunk loading keeps memory usage manageable.
parts = []
for chunk in pd.read_csv(PATH, usecols=USECOLS, chunksize=150_000, low_memory=False):
    chunk = chunk[chunk["loan_status"].isin(["Fully Paid","Charged Off","Default"])].copy()
    chunk["default"] = chunk["loan_status"].isin(["Charged Off","Default"]).astype("int8")
    parts.append(chunk)

df = pd.concat(parts, ignore_index=True)
df["issue_date"] = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce")
df["year"] = df["issue_date"].dt.year
df["term_num"] = pd.to_numeric(df["term"].str.extract(r"(\d+)")[0], errors="coerce")
df["emp_length_num"] = pd.to_numeric(
    df["emp_length"].replace({"< 1 year":"0.5","10+ years":"10"})
      .str.extract(r"(\d+\.?\d*)")[0], errors="coerce"
)
df["fico_mid"] = (df["fico_range_low"] + df["fico_range_high"]) / 2

print(df.shape)
print(df["default"].mean())

## 2. Target and leakage control

Only loans with observed final outcomes are used for the binary PD target:

- `Fully Paid` → 0
- `Charged Off`, `Default` → 1

Post-origination variables such as repayments, recoveries, last-payment fields, settlement fields and hardship fields are excluded from the PD model because they can leak future information.

In [ ]:
num = [
    "loan_amnt","int_rate","installment","annual_inc","dti","delinq_2yrs",
    "fico_mid","inq_last_6mths","open_acc","pub_rec","revol_bal","revol_util",
    "total_acc","pub_rec_bankruptcies","term_num","emp_length_num"
]
cat = ["grade","sub_grade","home_ownership","verification_status","purpose"]
features = num + cat

train = df[df.year <= 2016].copy()
valid = df[df.year == 2017].copy()
test = df[df.year == 2018].copy()

print("Train:", len(train))
print("2017 validation:", len(valid))
print("2018 test:", len(test))
print("2017 default rate:", valid.default.mean())
print("2018 default rate:", test.default.mean())

## 3. PD model — interpretable baseline

Logistic regression estimates:

\[
PD=P(Default=1|X)
\]

and

\[
\log\left(\frac{PD}{1-PD}\right)=\beta_0+\beta^TX
\]

It is deliberately used as an interpretable benchmark.

In [ ]:
# For a quick reproducible run, use a stratified development sample.
train_dev, _ = train_test_split(
    train, train_size=min(300_000, len(train)),
    stratify=train["default"], random_state=42
)

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), num),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat)
])

pd_model = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(solver="liblinear", max_iter=100))
])

pd_model.fit(train_dev[features], train_dev["default"])
p17 = pd_model.predict_proba(valid[features])[:,1]
p18 = pd_model.predict_proba(test[features])[:,1]

def ks_stat(y, p):
    order = np.argsort(-p)
    y = np.asarray(y)[order]
    return np.max(np.abs(
        np.cumsum(y)/y.sum() -
        np.cumsum(1-y)/(1-y).sum()
    ))

for name, y, p in [("2017", valid.default, p17), ("2018", test.default, p18)]:
    auc = roc_auc_score(y,p)
    print(name)
    print("AUC:", round(auc,4))
    print("KS:", round(ks_stat(y,p),4))
    print("Gini:", round(2*auc-1,4))
    print("Brier:", round(brier_score_loss(y,p),4))
    print("Mean PD:", round(p.mean(),4))

## 4. Calibration and backtesting

A good PD model must not only rank borrowers correctly; its probabilities should also be reasonably calibrated.

We compare:

\[
\text{Predicted PD}
\quad vs \quad
\text{Observed default rate}
\]

by risk decile.

In [ ]:
cal = pd.DataFrame({"pd":p17, "default":valid.default.to_numpy()})
cal["decile"] = pd.qcut(cal["pd"], 10, labels=False, duplicates="drop") + 1
calibration_table = cal.groupby("decile").agg(
    mean_predicted_pd=("pd","mean"),
    observed_default_rate=("default","mean"),
    observations=("default","size")
)
calibration_table["gap"] = (
    calibration_table["observed_default_rate"] -
    calibration_table["mean_predicted_pd"]
)
display(calibration_table)

plt.figure(figsize=(7,5))
plt.plot(calibration_table["mean_predicted_pd"],
         calibration_table["observed_default_rate"], marker="o")
plt.plot([0,1],[0,1], linestyle="--")
plt.xlabel("Mean predicted PD")
plt.ylabel("Observed default rate")
plt.title("2017 PD Calibration")
plt.show()

## 5. LGD

For defaulted loans, a simple realized LGD proxy is:

\[
LGD = \frac{FundedAmount-Recoveries}{FundedAmount}
\]

clipped to [0,1].

This is a **realized LGD analysis**, not a production LGD forecasting model. A future extension would build a separate predictive LGD model using only information available at the relevant prediction date.

In [ ]:
defaults17 = valid[valid.default == 1].copy()
defaults17["lgd_realized"] = (
    (defaults17["funded_amnt"] - defaults17["recoveries"]) /
    defaults17["funded_amnt"]
).clip(0,1)

mean_lgd = defaults17["lgd_realized"].mean()
print("Default observations:", len(defaults17))
print("Mean realized LGD:", round(mean_lgd,4))
print("Mean recovery rate:", round(1-mean_lgd,4))

## 6. EAD and Expected Credit Loss

For this academic project we use **funded amount as a simplified EAD proxy**.

For each loan:

\[
ECL_i = PD_i \times LGD \times EAD_i
\]

and:

\[
ECL_{portfolio} = \sum_i ECL_i
\]

This assumption must be disclosed because true EAD depends on the exposure outstanding at default and, for some products, future drawdowns.

In [ ]:
ecl_2017 = np.sum(
    p17 * mean_lgd * valid["funded_amnt"].to_numpy()
)
print("2017 funded exposure: $", round(valid["funded_amnt"].sum()))
print("Model-based ECL: $", round(ecl_2017))
print("ECL / exposure:", round(ecl_2017/valid["funded_amnt"].sum(),4))

## 7. Monte Carlo portfolio loss

To move from expected loss to tail risk, simulate correlated defaults.

A one-factor Gaussian structure is used:

\[
Z_i = \sqrt{\rho}F + \sqrt{1-\rho}\epsilon_i
\]

A borrower defaults when:

\[
Z_i < \Phi^{-1}(PD_i)
\]

Then portfolio loss is:

\[
L=\sum_i Default_i\times LGD_i\times EAD_i
\]

From simulated losses we calculate VaR and Expected Shortfall.

In [ ]:
rng = np.random.default_rng(42)

N = min(10_000, len(valid))
idx = rng.choice(len(valid), N, replace=False)

pd_port = p17[idx]
ead_port = valid["funded_amnt"].to_numpy()[idx]

rho = 0.15
SCENARIOS = 2_000

def simulate_losses(pd_vec, lgd, scenarios=SCENARIOS):
    factor = rng.normal(size=scenarios)
    eps = rng.normal(size=(scenarios, len(pd_vec)))
    latent = np.sqrt(rho)*factor[:,None] + np.sqrt(1-rho)*eps
    defaults = latent < norm.ppf(np.clip(pd_vec,1e-6,1-1e-6))[None,:]
    return (defaults * ead_port[None,:] * lgd).sum(axis=1)

base_losses = simulate_losses(pd_port, mean_lgd)

def var_es(losses, alpha):
    var = np.quantile(losses, alpha)
    es = losses[losses >= var].mean()
    return var, es

for a in [0.95,0.99]:
    v,e = var_es(base_losses,a)
    print(f"Base {a:.0%} VaR: ${v:,.0f} | ES: ${e:,.0f}")

## 8. Stress testing

Stress scenario:

- increase PD **odds by 50%**
- increase LGD by **5 percentage points**

The purpose is not to predict a particular crisis. It is to test **model sensitivity and portfolio resilience under adverse assumptions**.

In [ ]:
# Odds stress: odds' = 1.5 * odds
odds = pd_port / (1-pd_port)
stress_pd = (1.5*odds)/(1+1.5*odds)
stress_lgd = min(1.0, mean_lgd + 0.05)

stress_losses = simulate_losses(stress_pd, stress_lgd)

for a in [0.95,0.99]:
    v,e = var_es(stress_losses,a)
    print(f"Stress {a:.0%} VaR: ${v:,.0f} | ES: ${e:,.0f}")

## 9. Model-validation conclusions

The validation checklist is:

- **Discrimination:** ROC-AUC, KS, Gini
- **Probability quality:** Brier score and calibration
- **Temporal robustness:** out-of-time 2017/2018 evaluation
- **Benchmarking:** interpretable logistic baseline vs nonlinear benchmark
- **Backtesting:** predicted PD vs observed default rate by risk decile
- **Sensitivity:** change PD/LGD assumptions
- **Stress testing:** adverse portfolio scenario
- **Limitations:** EAD/LGD simplifications and dataset selection effects

# Extended Model Validation Suite

This extension uses the actual LendingClub data and adds: calibration intercept/slope, aggregate backtesting, decile backtesting with confidence intervals, vintage stability, PSI, ECL sensitivity, and Monte Carlo dependence sensitivity.

**Important:** the primary project headline results remain those from the original full-sample run. The reproducible extension below uses a 300,000-row random development sample from loans issued through 2016, while keeping the 2017 OOT validation population complete.


In [ ]:
# Extended validation: reproducible development sample + complete OOT populations
from scipy.stats import binom, norm
from scipy.special import logit, ndtri
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss

# df, features, num, cat should already exist from the original notebook
train_full = df[df.year <= 2016].copy()
train_ext = train_full.sample(n=min(300000, len(train_full)), random_state=42)
val17 = df[df.year == 2017].copy()
val18 = df[df.year == 2018].copy()

prep_ext = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat)
])
lr_ext = Pipeline([('prep', prep_ext), ('model', LogisticRegression(solver='lbfgs', max_iter=200, C=1.0))])
lr_ext.fit(train_ext[features], train_ext.default)

p17 = lr_ext.predict_proba(val17[features])[:,1]
y17 = val17.default.to_numpy()
print('2017 AUC:', roc_auc_score(y17,p17))
print('2017 Brier:', brier_score_loss(y17,p17))
print('2017 mean PD:', p17.mean())
print('2017 observed default:', y17.mean())

## Calibration and formal backtesting

In [ ]:
# Decile calibration + 95% binomial intervals
q=np.quantile(p17,np.linspace(0,1,11)); q[0]=0; q[-1]=1
bins=np.digitize(p17,q[1:-1],right=True)+1
cal=[]
for b in range(1,11):
    m=bins==b; n=m.sum(); obs=int(y17[m].sum()); rate=obs/n; pred=p17[m].mean()
    lo,hi=binom.interval(0.95,n,rate)
    cal.append([b,n,pred,rate,lo/n,hi/n,pred-rate])
calibration_2017=pd.DataFrame(cal,columns=['decile','n','pred_pd','obs_rate','ci_low','ci_high','pred_minus_obs'])
display(calibration_2017)

# Aggregate expected-vs-observed default backtest
expected=p17.sum(); observed=y17.sum(); variance=np.sum(p17*(1-p17))
z=(observed-expected)/np.sqrt(variance)
print('Expected defaults:',expected)
print('Observed defaults:',observed)
print('Observed - expected:',observed-expected)
print('Standardized backtest difference:',z)

# Calibration intercept and slope: logit(observed probability) on logit(model PD)
zpd=logit(np.clip(p17,1e-6,1-1e-6)).reshape(-1,1)
cal_model=LogisticRegression(solver='lbfgs').fit(zpd,y17)
print('Calibration intercept:',cal_model.intercept_[0])
print('Calibration slope:',cal_model.coef_[0,0])

## Vintage stability and Population Stability Index (PSI)

In [ ]:
vintage_rows=[]
for yr,d in [(2015,df[df.year==2015]),(2016,df[df.year==2016]),(2017,val17),(2018,val18)]:
    pp=lr_ext.predict_proba(d[features])[:,1]
    vintage_rows.append([yr,len(d),d.default.mean(),pp.mean(),roc_auc_score(d.default,pp),brier_score_loss(d.default,pp)])
vintage_validation=pd.DataFrame(vintage_rows,columns=['year','n','observed_default_rate','mean_pred_pd','auc','brier'])
display(vintage_validation)

# PSI of score distribution relative to 2016
base=lr_ext.predict_proba(df[df.year==2016][features])[:,1]
edges=np.unique(np.quantile(base,np.linspace(0,1,11)))
def score_dist(x):
    b=np.digitize(x,edges[1:-1],right=True); return np.bincount(b,minlength=len(edges)-1).astype(float)/len(x)
base_dist=score_dist(base)
psi_rows=[]
for yr in [2015,2016,2017,2018]:
    pp=lr_ext.predict_proba(df[df.year==yr][features])[:,1]
    d=score_dist(pp)
    psi=float(np.sum((d-base_dist)*np.log(np.clip(d,1e-6,None)/np.clip(base_dist,1e-6,None))))
    psi_rows.append([yr,psi])
psi_vintage=pd.DataFrame(psi_rows,columns=['year','PSI_vs_2016'])
display(psi_vintage)

## Actual-data ECL sensitivity

In [ ]:
# Use actual 2017 funded exposure and project LGD assumption
funded17=val17.funded_amnt.fillna(0).to_numpy()
lgd_base=0.9329
base_ecl=np.sum(p17*lgd_base*funded17)
rows=[]
for mult in [0.8,1.0,1.25,1.5]:
    ecl=np.sum(np.clip(p17*mult,0,1)*lgd_base*funded17)
    rows.append(['PD multiplier',mult,ecl,ecl/base_ecl-1])
for add in [-0.05,0,0.05]:
    lgd=np.clip(lgd_base+add,0,1); ecl=np.sum(p17*lgd*funded17)
    rows.append(['LGD shift',add,ecl,ecl/base_ecl-1])
for mult in [0.9,1.0,1.1,1.2]:
    ecl=np.sum(p17*lgd_base*funded17*mult)
    rows.append(['EAD multiplier',mult,ecl,ecl/base_ecl-1])
sensitivity=pd.DataFrame(rows,columns=['scenario_type','scenario_value','ecl','pct_change_vs_base'])
display(sensitivity)

## Monte Carlo dependence sensitivity

The same actual 10,000-loan 2017 sample and 2,000 scenarios can be used to test how the common-factor correlation assumption changes portfolio tail risk.

In [ ]:
rng=np.random.default_rng(42)
idx=rng.choice(len(val17),size=10000,replace=False)
pp=p17[idx]; ee=funded17[idx]; ll=np.full(len(idx),lgd_base)

def stress_odds(p,m=1.5):
    odds=p/(1-p); return (m*odds)/(1+m*odds)

def mc(rho,p,lgd,nsim=2000):
    F=rng.normal(size=nsim)
    eps=rng.normal(size=(nsim,len(p)))
    Z=np.sqrt(rho)*F[:,None]+np.sqrt(1-rho)*eps
    default=(Z < ndtri(np.clip(p,1e-9,1-1e-9))[None,:])
    loss=(default*(lgd*ee)[None,:]).sum(axis=1)
    q95,q99=np.quantile(loss,[.95,.99])
    return [loss.mean(),q95,q99,loss[loss>=q95].mean(),loss[loss>=q99].mean()]

rows=[]
for scenario,ps,lgs in [('Base',pp,ll),('Stress',stress_odds(pp,1.5),np.clip(ll+0.05,0,1))]:
    for rho in [0.0,0.10,0.15,0.30,0.50]:
        rows.append([scenario,rho,*mc(rho,ps,lgs)])
mc_sensitivity=pd.DataFrame(rows,columns=['scenario','rho','mean_loss','VaR95','VaR99','ES95','ES99'])
display(mc_sensitivity)

## Validation conclusion

The extended analysis now demonstrates actual-data **discrimination, calibration, out-of-time backtesting, benchmarking, vintage stability, PSI, financial sensitivity, stress testing, and dependence sensitivity**. The remaining production-level extensions would be formal regulatory backtesting across mature vintages, separately predicted LGD/EAD, macro-linked stress scenarios, and a documented model-governance framework.